# 🔥 QLoRA Fine-Tuning Project — Notebook 3: QLoRA Fine-Tuning
**Goal:** Fine-tune Mistral-7B using QLoRA on CNN/DailyMail training set

## What This Notebook Does
1. Loads processed dataset from Notebook 1
2. Sets up Mistral-7B with 4-bit quantization + LoRA adapters
3. Trains for 3 epochs with checkpoint saving every 100 steps
4. Evaluates on 1000 test samples
5. Saves model, predictions and ROUGE scores

## Checkpoint Resume System
- Saves every 100 steps automatically
- If session times out → just re-run all cells, it resumes automatically
- Checkpoints validated against current model to prevent mismatches

**Runtime:** ~11 hours (3 epochs)  
**Input:** `qlora-cnn-processed` dataset  
**Output:** `finetuned_results.csv`, `finetuned_predictions.csv`, trained model

In [ ]:
# ══════════════════════════════════════════════════════
# 📦 CELL 1: Install Dependencies
# ══════════════════════════════════════════════════════
!pip install -q bitsandbytes>=0.45.0 --upgrade
!pip install -q transformers>=4.40.0
!pip install -q datasets>=2.20.0        # 👈 no pin, use compatible version
!pip install -q accelerate>=0.28.0
!pip install -q peft>=0.10.0
!pip install -q evaluate>=0.4.1
!pip install -q rouge-score>=0.1.2
!pip install -q sentencepiece>=0.2.0

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("✅ Dependencies ready")

In [ ]:
# ══════════════════════════════════════════════════════
# ⚙️ CELL 2: Config
# ══════════════════════════════════════════════════════
import os, json, shutil, numpy as np, pandas as pd
import torch
import matplotlib.pyplot as plt
from datetime import datetime

os.environ["HF_DATASETS_CACHE"]  = "/kaggle/working/cache"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/working/cache"
os.makedirs("/kaggle/working/cache", exist_ok=True)

class Config:
    MODEL_ID         = "mistralai/Mistral-7B-v0.1"
    MAX_INPUT_LENGTH = 512
    BATCH_SIZE       = 2
    GRAD_ACCUM       = 8
    EPOCHS           = 3
    LR               = 2e-4
    LORA_R           = 8
    LORA_ALPHA       = 16
    LORA_DROPOUT     = 0.05
    LORA_TARGETS     = ["q_proj", "v_proj"]

# ── Paths ──────────────────────────────────────────────
INPUT_DATASET  = "/kaggle/input/datasets/pranitsaundankar/qlora-cnn-processed/qlora_project/processed_dataset"
BASE_PATH      = "/kaggle/working/qlora_project"
CHECKPOINT_DIR = f"{BASE_PATH}/checkpoints"
RESULTS_DIR    = f"{BASE_PATH}/results"
MODEL_DIR      = f"{BASE_PATH}/model"
DONE_FLAG      = f"{BASE_PATH}/TRAINING_COMPLETE"

for d in [CHECKPOINT_DIR, RESULTS_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"🔥 GPU: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"✅ Config ready")
print(f"   Model   : {Config.MODEL_ID}")
print(f"   Epochs  : {Config.EPOCHS}")
print(f"   Batch   : {Config.BATCH_SIZE} × {Config.GRAD_ACCUM} = {Config.BATCH_SIZE * Config.GRAD_ACCUM} effective")
print(f"   LR      : {Config.LR}")
print(f"   LoRA r  : {Config.LORA_R}")

if os.path.exists(DONE_FLAG):
    print("\n✅ TRAINING_COMPLETE flag found — training already finished in a previous session")
else:
    print("\n⏳ No completion flag — training will run (or resume from checkpoint)")

## ♻️ Checkpoint Detection

In [ ]:
# ══════════════════════════════════════════════════════
# ♻️ CELL 3: Detect & Validate Checkpoint
# ══════════════════════════════════════════════════════
def get_last_checkpoint(checkpoint_dir, model_id):
    """Find latest valid checkpoint for current model."""
    if not os.path.exists(checkpoint_dir):
        return None
    checkpoints = [
        os.path.join(checkpoint_dir, d)
        for d in os.listdir(checkpoint_dir)
        if d.startswith("checkpoint-") and d.split("-")[-1].isdigit()
    ]
    if not checkpoints:
        return None
    latest = max(checkpoints, key=lambda x: int(x.split("-")[-1]))
    adapter_cfg_path = os.path.join(latest, "adapter_config.json")
    if os.path.exists(adapter_cfg_path):
        with open(adapter_cfg_path) as f:
            cfg = json.load(f)
        saved_model = cfg.get("base_model_name_or_path", "")
        if saved_model and saved_model != model_id:
            print(f"⚠️  Checkpoint mismatch! Clearing...")
            shutil.rmtree(latest)
            return None
    return latest

RESUME_FROM = get_last_checkpoint(CHECKPOINT_DIR, Config.MODEL_ID)

# ── If working dir is empty (new session), restore from backup dataset ──
if not RESUME_FROM:
    backup_path      = "/kaggle/input/datasets/pranitsaundankar/qlora-checkpoint-backup/checkpoint_backup"
    resume_info_path = "/kaggle/input/datasets/pranitsaundankar/qlora-checkpoint-backup/resume_info.json"
    if os.path.exists(backup_path):
        print(f"📦 No local checkpoint — restoring from backup dataset...")
        restored = os.path.join(CHECKPOINT_DIR, "checkpoint-restored")
        if os.path.exists(restored):
            shutil.rmtree(restored)
        shutil.copytree(backup_path, restored)
        RESUME_FROM = restored
        if os.path.exists(resume_info_path):
            with open(resume_info_path) as f:
                info = json.load(f)
            print(f"   Last step   : {info['last_step']}")
            print(f"   Epochs done : {info['epochs_done']:.2f}")
            print(f"   Saved at    : {info['saved_at']}")
    else:
        print("🆕 No checkpoint or backup found — starting fresh")

if RESUME_FROM:
    last_part = RESUME_FROM.split("-")[-1]
    step_display = last_part if last_part.isdigit() else "restored from backup"
    print(f"🔁 Will resume from: {step_display}")
    print(f"   Path: {RESUME_FROM}")
else:
    print("🆕 Starting fresh")

## 📥 Load Dataset

In [ ]:
# ══════════════════════════════════════════════════════
# 📥 CELL 4: Load Dataset
# ══════════════════════════════════════════════════════
from datasets import load_from_disk

print(f"📥 Loading dataset...")
dataset = load_from_disk(INPUT_DATASET)

train_dataset = dataset["train"]
val_dataset   = dataset["validation"]
test_dataset  = dataset["test"]

print(f"✅ Dataset loaded")
print(f"   Train : {len(train_dataset):,}")
print(f"   Val   : {len(val_dataset):,}")
print(f"   Test  : {len(test_dataset):,}")
print(f"   Cols  : {train_dataset.column_names}")

## 🔤 Load Tokenizer

In [ ]:
# ══════════════════════════════════════════════════════
# 🔤 CELL 5: Load Tokenizer
# ══════════════════════════════════════════════════════
from transformers import AutoTokenizer

print(f"Loading tokenizer: {Config.MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_ID, use_fast=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"
print("✅ Tokenizer loaded")

## 🔥 Format & Tokenize Dataset

In [ ]:
# ══════════════════════════════════════════════════════
# 🔥 CELL 6: Format & Tokenize
# ══════════════════════════════════════════════════════
def format_and_tokenize(example):
    prompt = (
        "### Instruction:\nSummarize the following news article concisely.\n\n"
        f"### Input:\n{example['input']}\n\n### Response:\n"
    )
    full_text = prompt + example['output'] + tokenizer.eos_token

    tokens = tokenizer(
        full_text,
        truncation=True,
        padding="max_length",
        max_length=Config.MAX_INPUT_LENGTH
    )

    # Mask prompt tokens so loss is only on the response
    prompt_tokens = tokenizer(prompt, truncation=True, max_length=Config.MAX_INPUT_LENGTH)
    prompt_len = len(prompt_tokens["input_ids"])

    labels = tokens["input_ids"].copy()
    labels[:prompt_len] = [-100] * prompt_len
    tokens["labels"] = labels
    return tokens

print("🔄 Tokenizing train...")
train_tokenized = train_dataset.map(
    format_and_tokenize,
    remove_columns=train_dataset.column_names,
    batched=False,
    load_from_cache_file=False,
    keep_in_memory=True,
    desc="Tokenizing train"
)

print("🔄 Tokenizing val...")
val_tokenized = val_dataset.map(
    format_and_tokenize,
    remove_columns=val_dataset.column_names,
    batched=False,
    load_from_cache_file=False,
    keep_in_memory=True,
    desc="Tokenizing val"
)

print(f"\n✅ Tokenization done")
print(f"   Token length: {len(train_tokenized[0]['input_ids'])} (expected {Config.MAX_INPUT_LENGTH})")

## 🤖 Load Model with QLoRA

In [ ]:
# ══════════════════════════════════════════════════════
# 🤖 CELL 7: Load Model with QLoRA
# ══════════════════════════════════════════════════════
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print(f"Loading {Config.MODEL_ID}...")
model = AutoModelForCausalLM.from_pretrained(
    Config.MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

lora_config = LoraConfig(
    r=Config.LORA_R,
    lora_alpha=Config.LORA_ALPHA,
    target_modules=Config.LORA_TARGETS,
    lora_dropout=Config.LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("✅ QLoRA model ready")

## 🚀 Train

In [ ]:
# ══════════════════════════════════════════════════════
# 🚀 CELL 8: Setup Trainer & Train
# ══════════════════════════════════════════════════════
from transformers import (
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling, TrainerCallback
)

# ── Skip if already complete ───────────────────────────
if os.path.exists(DONE_FLAG):
    print("✅ Training already complete — skipping training")
    print("   Loading model from saved adapter weights...")
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    from peft import PeftModel

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        Config.MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto"
    )
    model = PeftModel.from_pretrained(base_model, MODEL_DIR)
    model.eval()
    print("✅ Model loaded from saved adapter weights")

else:
    class CheckpointCallback(TrainerCallback):
        def on_save(self, args, state, control, **kwargs):
            step = state.global_step
            ckpt = os.path.join(args.output_dir, f"checkpoint-{step}")
            print(f"\n💾 Checkpoint saved at step {step} → {ckpt}")

        def on_log(self, args, state, control, logs=None, **kwargs):
            if logs:
                step = state.global_step
                loss = logs.get("loss", "")
                eval_loss = logs.get("eval_loss", "")
                if loss:
                    print(f"   Step {step:4d} | Loss: {loss:.4f}" +
                          (f" | Eval Loss: {eval_loss:.4f}" if eval_loss else ""))

    class CheckpointBackupCallback(TrainerCallback):
        """
        After every checkpoint save, backs up the latest checkpoint
        to /kaggle/working/checkpoint_backup so you can commit it
        as a dataset and resume in a new session if needed.
        """
        def on_save(self, args, state, control, **kwargs):
            step = state.global_step
            src  = os.path.join(args.output_dir, f"checkpoint-{step}")
            dst  = "/kaggle/working/checkpoint_backup"

            if not os.path.exists(src):
                return

            # Replace previous backup with latest checkpoint
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)

            # Save training history snapshot inside backup
            history_snapshot = os.path.join(dst, "training_history_snapshot.json")
            with open(history_snapshot, "w") as f:
                json.dump(state.log_history, f, indent=2)

            # Save resume info so next session knows exactly where to start
            resume_info = {
                "last_step":    state.global_step,
                "epochs_done":  state.epoch,
                "saved_at":     datetime.now().isoformat(),
            }
            with open("/kaggle/working/resume_info.json", "w") as f:
                json.dump(resume_info, f, indent=2)

            print(f"   📦 Backup saved → {dst} (step {step})")

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    training_args = TrainingArguments(
        output_dir=CHECKPOINT_DIR,
        per_device_train_batch_size=Config.BATCH_SIZE,
        gradient_accumulation_steps=Config.GRAD_ACCUM,
        num_train_epochs=Config.EPOCHS,
        learning_rate=Config.LR,

        gradient_checkpointing=True,
        optim="adamw_bnb_8bit",
        dataloader_num_workers=2,
        dataloader_pin_memory=False,

        logging_steps=50,
        logging_first_step=True,

        save_strategy="steps",
        save_steps=200,
        save_total_limit=3,

        eval_strategy="steps",
        eval_steps=200,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        fp16=True,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
        data_collator=data_collator,
        callbacks=[CheckpointCallback(), CheckpointBackupCallback()]
    )

    if RESUME_FROM:
        print(f"\n🔁 Resuming from {RESUME_FROM}")
        trainer.train(resume_from_checkpoint=RESUME_FROM)
    else:
        print("\n🆕 Starting fresh training")
        trainer.train()

    # ── Save final model ───────────────────────────────
    trainer.save_model(MODEL_DIR)
    tokenizer.save_pretrained(MODEL_DIR)

    # ── Write completion flag ──────────────────────────
    with open(DONE_FLAG, "w") as f:
        json.dump({
            "completed_at":    datetime.now().isoformat(),
            "final_step":      trainer.state.global_step,
            "best_checkpoint": trainer.state.best_model_checkpoint
        }, f, indent=2)

    print("\n✅ Training complete — model saved")
    print(f"✅ TRAINING_COMPLETE flag written")

    # ── Save training history ──────────────────────────
    history = trainer.state.log_history
    with open(os.path.join(RESULTS_DIR, "training_history.json"), "w") as f:
        json.dump(history, f, indent=2)
    print(f"✅ training_history.json saved")

## 📈 Training Curves

In [ ]:
# ══════════════════════════════════════════════════════
# 📈 CELL 9: Plot Training Curves
# ══════════════════════════════════════════════════════

# Load from file if trainer not in memory (resumed session)
history_path = os.path.join(RESULTS_DIR, "training_history.json")
if 'trainer' in dir() and hasattr(trainer, 'state'):
    history = trainer.state.log_history
    print("📈 Using in-memory training history")
elif os.path.exists(history_path):
    with open(history_path) as f:
        history = json.load(f)
    print("📈 Loaded training history from file")
else:
    # Try backup snapshot
    snapshot = "/kaggle/working/checkpoint_backup/training_history_snapshot.json"
    if os.path.exists(snapshot):
        with open(snapshot) as f:
            history = json.load(f)
        print("📈 Loaded training history from checkpoint snapshot")
    else:
        print("⚠️  No training history found — skipping curves")
        history = []

if history:
    train_steps, train_losses = [], []
    eval_steps,  eval_losses  = [], []

    for log in history:
        if "loss" in log and "eval_loss" not in log:
            train_steps.append(log["step"])
            train_losses.append(log["loss"])
        if "eval_loss" in log:
            eval_steps.append(log["step"])
            eval_losses.append(log["eval_loss"])

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(train_steps, train_losses, label="Train Loss", color="steelblue", linewidth=2)
    if eval_losses:
        ax.plot(eval_steps, eval_losses, label="Val Loss", color="coral",
                linewidth=2, marker='o', markersize=4)
    ax.set_xlabel("Training Steps", fontsize=12)
    ax.set_ylabel("Loss", fontsize=12)
    ax.set_title("QLoRA Training Curves — Mistral-7B on CNN/DailyMail", fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(alpha=0.3)
    plt.tight_layout()

    curves_path = os.path.join(RESULTS_DIR, "training_curves.png")
    plt.savefig(curves_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved training_curves.png")

## 🎯 Evaluate on Test Set

In [ ]:
# ══════════════════════════════════════════════════════
# 🎯 CELL 10: Generate Predictions on Test Set
# ══════════════════════════════════════════════════════
from tqdm import tqdm

print("🎯 Generating predictions on test set...")
print(f"   Samples: {len(test_dataset)}")

# Disable gradient checkpointing before inference
model.gradient_checkpointing_disable()
model.config.use_cache = True
tokenizer.padding_side = "left"

INFER_BATCH_SIZE = 4

def generate_batch(texts, batch_size=INFER_BATCH_SIZE):
    all_preds = []
    prompts = [
        f"### Instruction:\nSummarize the following news article concisely.\n\n"
        f"### Input:\n{text}\n\n### Response:\n"
        for text in texts
    ]

    for i in tqdm(range(0, len(prompts), batch_size), desc="Generating"):
        batch = prompts[i:i+batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=Config.MAX_INPUT_LENGTH
        ).to(model.device)

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=True
            )

        for j, output in enumerate(outputs):
            input_len = inputs["attention_mask"][j].sum().item()
            pred = tokenizer.decode(output[input_len:], skip_special_tokens=True)
            pred = pred.split("\n")[0].strip()
            all_preds.append(pred)

        torch.cuda.empty_cache()

    return all_preds

# Fast column access
test_texts = test_dataset["input"]
references = test_dataset["output"]

finetuned_preds = generate_batch(test_texts)

print(f"\n✅ Generated {len(finetuned_preds)} predictions")
print(f"\nSample prediction: {finetuned_preds[0]}")

In [ ]:
# ══════════════════════════════════════════════════════
# 📊 CELL 11: Compute ROUGE & Save Results
# ══════════════════════════════════════════════════════
import evaluate

rouge = evaluate.load("rouge")
scores = rouge.compute(predictions=finetuned_preds, references=references)

print("📊 Fine-tuned Model ROUGE Scores:")
print(f"   ROUGE-1  : {scores['rouge1']:.4f}")
print(f"   ROUGE-2  : {scores['rouge2']:.4f}")
print(f"   ROUGE-L  : {scores['rougeL']:.4f}")
print(f"   ROUGE-Lsum: {scores['rougeLsum']:.4f}")

# ── Save finetuned results CSV ─────────────────────────
finetuned_results = pd.DataFrame([{
    "method":     "QLoRA Fine-tuned",
    "rouge1":     scores["rouge1"],
    "rouge2":     scores["rouge2"],
    "rougeL":     scores["rougeL"],
    "rougeLsum":  scores["rougeLsum"]
}])
results_path = os.path.join(RESULTS_DIR, "finetuned_results.csv")
finetuned_results.to_csv(results_path, index=False)
print(f"\n✅ Saved finetuned_results.csv")

# ── Save predictions CSV ───────────────────────────────
preds_df = pd.DataFrame({
    "reference":  references,
    "finetuned":  finetuned_preds
})
preds_path = os.path.join(RESULTS_DIR, "finetuned_predictions.csv")
preds_df.to_csv(preds_path, index=False)
print(f"✅ Saved finetuned_predictions.csv ({len(preds_df)} rows)")

In [ ]:
# ══════════════════════════════════════════════════════
# 📊 CELL 12: Save Full Training Summary
# ══════════════════════════════════════════════════════
rouge_scores_summary = {}
if 'scores' in dir():
    rouge_scores_summary = {
        "rouge1":    scores["rouge1"],
        "rouge2":    scores["rouge2"],
        "rougeL":    scores["rougeL"],
        "rougeLsum": scores["rougeLsum"]
    }
else:
    print("⚠️  ROUGE scores not found — Cell 11 may not have run. Saving summary without scores.")

summary = {
    "model":            Config.MODEL_ID,
    "training_date":    datetime.now().strftime("%Y-%m-%d"),
    "dataset":          "cnn_dailymail",
    "epochs":           Config.EPOCHS,
    "batch_size":       Config.BATCH_SIZE,
    "grad_accum":       Config.GRAD_ACCUM,
    "effective_batch":  Config.BATCH_SIZE * Config.GRAD_ACCUM,
    "learning_rate":    Config.LR,
    "max_input_length": Config.MAX_INPUT_LENGTH,
    "lora_r":           Config.LORA_R,
    "lora_alpha":       Config.LORA_ALPHA,
    "lora_targets":     Config.LORA_TARGETS,
    "train_samples":    len(train_dataset),
    "test_samples":     len(test_dataset),
    "rouge_scores":     rouge_scores_summary
}

summary_path = os.path.join(RESULTS_DIR, "training_summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("📊 Training Summary:")
print(json.dumps(summary, indent=2))
print(f"\n✅ Saved training_summary.json")

In [ ]:
# ══════════════════════════════════════════════════════
# 🔍 CELL 13: Qualitative Examples
# ══════════════════════════════════════════════════════
print("🔍 Sample Predictions\n")
print("=" * 80)
for i in range(5):
    print(f"\n📌 Example {i+1}")
    print(f"ARTICLE   : {test_texts[i][:200]}...")
    print(f"REFERENCE : {references[i]}")
    print(f"PREDICTED : {finetuned_preds[i]}")
    print("─" * 80)

## ✅ Notebook 3 Complete!

### ROUGE Scores Obtained — see training_summary.json

### What was saved:
| File | Description |
|------|-------------|
| `model/` | Fine-tuned LoRA adapter weights |
| `finetuned_results.csv` | ROUGE scores |
| `finetuned_predictions.csv` | All 1000 predictions + references |
| `training_curves.png` | Loss curves |
| `training_history.json` | Full training log |
| `training_summary.json` | Complete experiment summary |